# Customer churn prediction

This notebook compares models on the Telco Customer Churn dataset, tunes Random Forest and XGBoost with Optuna, and uses SHAP to inspect predictions.

I use ROC-AUC for model selection and keep 20% of the data aside for the final test. The random seed is 42. Run the notebook from top to bottom with Customer_Churn.csv in the repository root.

## 1. Setup and data inspection

In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
import shap
from IPython.display import display
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

RANDOM_STATE = 42
N_TRIALS = 30
sns.set_theme(style="whitegrid", palette="muted")
optuna.logging.set_verbosity(optuna.logging.WARNING)

candidates = [
    Path("data/Customer_Churn.csv"),
    Path("../data/Customer_Churn.csv"),
    Path("Customer_Churn.csv"),
    Path("../Customer_Churn.csv"),
]
DATA_PATH = next((path for path in candidates if path.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Place Customer_Churn.csv in data/ or next to this notebook.")

PROJECT_DIR = DATA_PATH.parent.parent if DATA_PATH.parent.name == "data" else DATA_PATH.parent
MODELS_DIR = PROJECT_DIR
REPORTS_DIR = PROJECT_DIR / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
for directory in (REPORTS_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Dataset: {DATA_PATH.resolve()}")

In [ ]:
data = pd.read_csv(DATA_PATH)
print(f"Shape: {data.shape}")
display(data.head())
print("Missing values before cleaning:")
display(data.isna().sum().loc[lambda s: s > 0])
print("Churn distribution (%):")
display((data["Churn"].value_counts(normalize=True) * 100).round(2))

# Blank strings in TotalCharges are not reported as missing before conversion.
total_charges = pd.to_numeric(data["TotalCharges"], errors="coerce")
print(f"Invalid or blank TotalCharges: {total_charges.isna().sum()}")

## 2. Exploratory data analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
sns.countplot(data=data, x="Churn", order=["No", "Yes"], color="#4C72B0", ax=axes[0])
axes[0].set_title("Customer churn distribution")
axes[0].set_ylabel("Customers")

sns.countplot(data=data, x="Contract", hue="Churn", palette={"No": "#4C72B0", "Yes": "#C44E52"}, ax=axes[1])
axes[1].set_title("Churn by contract type")
axes[1].tick_params(axis="x", rotation=18)

sns.boxplot(data=data, x="Churn", y="MonthlyCharges", color="#4C72B0", ax=axes[2])
axes[2].set_title("Monthly charges by churn")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "churn_eda.png", dpi=160, bbox_inches="tight")
plt.show()

## 3. Preprocessing

Yes/no fields and contract type use fixed mappings. Missing values, scaling and one-hot encoding are handled inside the sklearn pipeline, so each CV fold learns preprocessing from its own training data.

In [ ]:
service_cols = [
    "MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies",
]
binary_map = {
    "gender": {"Female": 0, "Male": 1},
    "Partner": {"No": 0, "Yes": 1},
    "Dependents": {"No": 0, "Yes": 1},
    "PhoneService": {"No": 0, "Yes": 1},
    "PaperlessBilling": {"No": 0, "Yes": 1},
    "Churn": {"No": 0, "Yes": 1},
}
binary_map.update({col: {"No": 0, "Yes": 1} for col in service_cols})
contract_map = {"Month-to-month": 0, "One year": 1, "Two year": 2}

prepared = data.copy()
prepared["TotalCharges"] = pd.to_numeric(prepared["TotalCharges"], errors="coerce")
for col in service_cols:
    prepared[col] = prepared[col].replace({
        "No internet service": "No", "No phone service": "No"
    })
for col, mapping in {**binary_map, "Contract": contract_map}.items():
    unknown = set(prepared[col].dropna().unique()) - set(mapping)
    if unknown:
        raise ValueError(f"Unexpected values in {col}: {unknown}")
    prepared[col] = prepared[col].map(mapping)

X = prepared.drop(columns=["customerID", "Churn"])
y = prepared["Churn"].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape}; test: {X_test.shape}")
print(f"Churn rate — train: {y_train.mean():.3f}; test: {y_test.mean():.3f}")

In [ ]:
numeric_features = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_features = ["InternetService", "PaymentMethod"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
        ]), numeric_features),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False),
         categorical_features),
    ],
    remainder="passthrough",
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def make_pipeline(classifier):
    return Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", classifier),
    ])

## 4. Model comparison

Five-fold stratified cross-validation on the training set. The test set is not used here.

In [ ]:
models = {
    "LogReg L2": LogisticRegression(
        penalty="l2", C=1.0, max_iter=1000,
        class_weight="balanced", random_state=RANDOM_STATE,
    ),
    "LogReg L2 (strong)": LogisticRegression(
        penalty="l2", C=0.01, max_iter=1000,
        class_weight="balanced", random_state=RANDOM_STATE,
    ),
    "LogReg L1": LogisticRegression(
        penalty="l1", solver="liblinear", C=1.0, max_iter=1000,
        class_weight="balanced", random_state=RANDOM_STATE,
    ),
    "LogReg ElasticNet": LogisticRegression(
        penalty="elasticnet", solver="saga", l1_ratio=0.5, C=1.0,
        max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE,
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=300, max_depth=10, min_samples_leaf=5,
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=1,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        scale_pos_weight=3.0, random_state=RANDOM_STATE,
        eval_metric="logloss", n_jobs=1, tree_method="hist",
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=300, learning_rate=0.05, num_leaves=31,
        class_weight="balanced", random_state=RANDOM_STATE,
        verbose=-1, n_jobs=1,
    ),
}

scoring = {
    "roc_auc": "roc_auc", "f1": "f1", "precision": "precision",
    "recall": "recall", "accuracy": "accuracy",
}
cv_rows = []
for name, classifier in models.items():
    scores = cross_validate(
        make_pipeline(classifier), X_train, y_train,
        cv=cv, scoring=scoring, n_jobs=-1,
    )
    cv_rows.append({
        "model": name,
        "roc_auc_mean": scores["test_roc_auc"].mean(),
        "roc_auc_std": scores["test_roc_auc"].std(),
        "f1_mean": scores["test_f1"].mean(),
        "precision_mean": scores["test_precision"].mean(),
        "recall_mean": scores["test_recall"].mean(),
        "accuracy_mean": scores["test_accuracy"].mean(),
    })

cv_summary = pd.DataFrame(cv_rows).sort_values("roc_auc_mean", ascending=False)
display(cv_summary.round(4))

In [ ]:
# Check the L1 coefficients.
l1_pipeline = make_pipeline(LogisticRegression(
    penalty="l1", solver="liblinear", C=0.5,
    max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE,
))
l1_pipeline.fit(X_train, y_train)
feature_names = [name.split("__", 1)[-1] for name in
                 l1_pipeline.named_steps["preprocessor"].get_feature_names_out()]
coefficients = pd.DataFrame({
    "feature": feature_names,
    "coefficient": l1_pipeline.named_steps["classifier"].coef_[0],
})
coefficients["abs_coefficient"] = coefficients["coefficient"].abs()
coefficients = coefficients.sort_values("abs_coefficient", ascending=False)
display(coefficients)
print(f"Zero coefficients: {(coefficients['coefficient'] == 0).sum()}/{len(coefficients)}")

## 5. Optuna tuning

I tune Random Forest and XGBoost with the same CV setup. Change N_TRIALS for a faster test run.

In [ ]:
def objective_rf(trial):
    classifier = RandomForestClassifier(
        n_estimators=350,
        max_depth=trial.suggest_int("max_depth", 4, 20),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 20),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 20),
        max_features=trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=1,
    )
    return cross_val_score(
        make_pipeline(classifier), X_train, y_train,
        cv=cv, scoring="roc_auc", n_jobs=-1,
    ).mean()

study_rf = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
study_rf.optimize(objective_rf, n_trials=N_TRIALS)
print(f"RandomForest best CV ROC-AUC: {study_rf.best_value:.4f}")
print("Best parameters:", study_rf.best_params)

In [ ]:
def objective_xgb(trial):
    classifier = XGBClassifier(
        n_estimators=trial.suggest_int("n_estimators", 100, 600, step=50),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        max_depth=trial.suggest_int("max_depth", 3, 12),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
        gamma=trial.suggest_float("gamma", 0.0, 1.0),
        scale_pos_weight=3.0, random_state=RANDOM_STATE,
        eval_metric="logloss", n_jobs=1, tree_method="hist",
    )
    return cross_val_score(
        make_pipeline(classifier), X_train, y_train,
        cv=cv, scoring="roc_auc", n_jobs=-1,
    ).mean()

study_xgb = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS)
print(f"XGBoost best CV ROC-AUC: {study_xgb.best_value:.4f}")
print("Best parameters:", study_xgb.best_params)

In [ ]:
tuned_summary = pd.DataFrame([
    {"model": "RandomForest (Optuna)", "roc_auc_cv": study_rf.best_value},
    {"model": "XGBoost (Optuna)", "roc_auc_cv": study_xgb.best_value},
]).sort_values("roc_auc_cv", ascending=False)
display(tuned_summary.round(4))

cv_summary.to_csv(REPORTS_DIR / "baseline_cv_results.csv", index=False)
tuned_summary.to_csv(REPORTS_DIR / "tuned_cv_results.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=cv_summary, y="model", x="roc_auc_mean", color="#4C72B0", ax=ax)
ax.set(title="Baseline model comparison (5-fold CV)", xlabel="Mean ROC-AUC", ylabel="")
ax.set_xlim(0, 1)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "model_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

## 6. Test-set results

I keep the tuned XGBoost as the deployed model, as in the initial version of this project. Compare its CV score against the other models above before drawing conclusions. Test metrics are reported once.

In [ ]:
final_model = make_pipeline(XGBClassifier(
    **study_xgb.best_params,
    scale_pos_weight=3.0,
    random_state=RANDOM_STATE,
    eval_metric="logloss",
    n_jobs=1,
    tree_method="hist",
))
final_model.fit(X_train, y_train)

pred_proba = final_model.predict_proba(X_test)[:, 1]
pred_label = final_model.predict(X_test)
test_metrics = {
    "roc_auc": float(roc_auc_score(y_test, pred_proba)),
    "f1_churn": float(f1_score(y_test, pred_label)),
    "precision_churn": float(precision_score(y_test, pred_label)),
    "recall_churn": float(recall_score(y_test, pred_label)),
    "accuracy": float(accuracy_score(y_test, pred_label)),
}
display(pd.DataFrame([test_metrics]).round(4))
print(classification_report(y_test, pred_label, target_names=["No churn", "Churn"]))
with open(REPORTS_DIR / "test_metrics.json", "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, indent=2)

## 7. SHAP

SHAP values for this XGBoost model are in raw score units (log-odds), not percentage points.

In [ ]:
fitted_preprocessor = final_model.named_steps["preprocessor"]
fitted_classifier = final_model.named_steps["classifier"]
X_test_transformed = fitted_preprocessor.transform(X_test)
feature_names = [name.split("__", 1)[-1] for name in
                 fitted_preprocessor.get_feature_names_out()]

explainer = shap.TreeExplainer(fitted_classifier)
shap_values = explainer.shap_values(X_test_transformed)
if isinstance(shap_values, list):
    shap_values = shap_values[-1]
shap_values = np.asarray(shap_values)
if shap_values.ndim == 3:
    shap_values = shap_values[:, :, 1]
base_value = float(np.asarray(explainer.expected_value).reshape(-1)[-1])
print(f"SHAP values: {shap_values.shape}; features: {len(feature_names)}")

In [ ]:
shap.summary_plot(
    shap_values, X_test_transformed,
    feature_names=feature_names, plot_type="bar",
    plot_size=(10, 6), show=False,
)
plt.title("Mean absolute SHAP importance")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "shap_importance.png", dpi=160, bbox_inches="tight")
plt.show()

shap.summary_plot(
    shap_values, X_test_transformed,
    feature_names=feature_names,
    plot_size=(10, 6), show=False,
)
plt.title("SHAP impact on churn prediction")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "shap_summary.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
high_risk_idx = int(np.argmax(pred_proba))
print(f"Test row: {high_risk_idx}")
print(f"Predicted churn probability: {pred_proba[high_risk_idx]:.4f}")
print(f"Observed churn: {y_test.iloc[high_risk_idx]}")

explanation = shap.Explanation(
    values=shap_values[high_risk_idx],
    base_values=base_value,
    data=X_test_transformed[high_risk_idx],
    feature_names=feature_names,
)
shap.plots.waterfall(explanation, max_display=15, show=False)
plt.title("High-risk customer explanation")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "shap_waterfall.png", dpi=160, bbox_inches="tight")
plt.show()

## 8. Save model and results

The model is stored under both existing filenames because the API uses churn_model.joblib. encodings.json holds the fixed mappings used before the sklearn pipeline. InternetService and PaymentMethod are one-hot encoded inside the pipeline.

In [ ]:

joblib.dump(final_model, MODELS_DIR / "xgb_churn_model.joblib")
joblib.dump(final_model, MODELS_DIR / "churn_model.joblib")

encodings = {
    "binary_map": binary_map,
    "contract_map": contract_map,
    "internet_service_map": {"DSL": 0, "Fiber optic": 1, "No": 2},
    "payment_method_map": {
        "Electronic check": 0,
        "Mailed check": 1,
        "Bank transfer (automatic)": 2,
        "Credit card (automatic)": 3,
    },
}
with open(MODELS_DIR / "encodings.json", "w", encoding="utf-8") as f:
    json.dump(encodings, f, indent=2)

print("Saved models:", MODELS_DIR / "xgb_churn_model.joblib", "and", MODELS_DIR / "churn_model.joblib")
print("Saved encodings:", MODELS_DIR / "encodings.json")
print("Saved reports and figures:", REPORTS_DIR)

# Check that the saved model gives the same predictions.
restored_model = joblib.load(MODELS_DIR / "churn_model.joblib")
np.testing.assert_allclose(
    restored_model.predict_proba(X_test.iloc[:5]),
    final_model.predict_proba(X_test.iloc[:5]),
)
print("Saved model reload check passed.")